# schema-bridge — matcher test

Tests the full pipeline step by step:
1. Parse both model files into Python dicts
2. Extract subgraphs for concepts of interest
3. Run rapidfuzz lexical matching
4. Inspect results

Run cells top to bottom. Each cell depends on the ones above it.

## setup

Add `etl/` folder to the Python path so imports work from the notebook.

In [1]:
import pathlib

# current directory when jupyter runs — reliable in VS Code notebooks
ROOT = pathlib.Path().resolve()

# if that doesn't point to your project root, hardcode it:
# ROOT = pathlib.Path("C:/Users/........")

GQL_PATH = ROOT / "data_models" / "ilap_graphql_schema.graphql"
TTL_PATH = ROOT / "data_models" / "sdo.ttl"

print("ROOT    :", ROOT)
print("GQL     :", GQL_PATH, "exists:", GQL_PATH.exists())
print("TTL     :", TTL_PATH, "exists:", TTL_PATH.exists())

ROOT    : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test
GQL     : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test\data_models\ilap_graphql_schema.graphql exists: True
TTL     : C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test\data_models\sdo.ttl exists: True


## parse both files

In [2]:
from etl.extractor import parse_graphql, parse_sdo

GQL_PATH = ROOT / "data_models" / "ilap_graphql_schema.graphql"
TTL_PATH = ROOT / "data_models" / "sdo.ttl"

graphql_index = parse_graphql(str(GQL_PATH))
sdo_index     = parse_sdo(str(TTL_PATH))

print(f"GraphQL index : {len(graphql_index)} entries")
print(f"SDO index     : {len(sdo_index)} entries")

GraphQL index : 246 entries
SDO index     : 157 entries


### spot-check — look at one entry from each

In [3]:
import pprint

# GraphQL — Activity type (fields truncated for readability)
act = graphql_index["Activity"]
print(f"Activity — kind: {act['kind']}, fields: {len(act['fields'])}")

# show first 5 fields
for name, field in list(act["fields"].items())[:5]:
    print(f"  {name:<30} type={field['type']:<20} is_relation={field['is_relation']}")

Activity — kind: object, fields: 278
  predecessors                   type=Successor            is_relation=True
  resourceAssignments            type=ResourceAssignment   is_relation=True
  resourceAvailabilities         type=ResourceAvailability is_relation=True
  successors                     type=Successor            is_relation=True
  calendar                       type=Calendar             is_relation=True


In [4]:
# SDO — ScheduleActivity class
sa = sdo_index["ScheduleActivity"]
print(f"ScheduleActivity")
print(f"  definition  : {sa['definition'][:100]}")
print(f"  subclass_of : {sa['subclass_of']}")
print(f"  subclasses  : {sa['subclasses']}")
print(f"  properties  : {sa['properties'][:8]} ...")

ScheduleActivity
  definition  : An activity that identifies a piece of work that is required to be undertaken to complete a schedule
  subclass_of : ['Activity']
  subclasses  : ['CancelledActivity', 'AlwaysOnScheduleActivity', 'EarlyFinishActivity', 'EarlyStartActivity', 'LateFinishActivity', 'LateStartActivity']
  properties  : ['actualFinish', 'actualStart', 'earlyFinish', 'earlyStart', 'finishNoEarlierThan', 'finishNoLaterThan', 'freeFloat', 'hasBreakdownStructureElement'] ...


## extract concept subgraphs

Specify your seeds here. The walker finds everything reachable from them.

In [5]:
from etl.concepts import extract, describe

# change these to scope to different concepts
SEEDS = ["Activity", "Schedule", "Project", "Data", "ScheduleActivity"]

gql_subgraph = extract(graphql_index, SEEDS)
sdo_subgraph = extract(sdo_index,     SEEDS)

print(f"GraphQL subgraph : {len(gql_subgraph)} nodes")
print(f"SDO subgraph     : {len(sdo_subgraph)} nodes")

  [concepts] Warning: seed 'Project' not found in index — skipped
  [concepts] Warning: seed 'ScheduleActivity' not found in index — skipped
  [concepts] Warning: seed 'Activity' not found in index — skipped
  [concepts] Warning: seed 'Data' not found in index — skipped
GraphQL subgraph : 75 nodes
SDO subgraph     : 104 nodes


In [6]:
# what did the walker pull in?
describe(gql_subgraph, label="GraphQL")


════════════════════════════════════════════════════════════
  GraphQL subgraph  —  75 nodes
════════════════════════════════════════════════════════════

By node type:
  attribute          (  1)  UUID
  concept            ( 52)  ActivityMetadataFieldValue, ActivityStructureElement, Calendar, CalendarOperation, ConnectedPeriod, IlapFile...
  enum               ( 19)  ActivityType, DayOfWeek, ImportStatus, OperationType, PlanningLevel, PlanningObjectTypes...
  seed               (  3)  Activity, Data, Schedule

By hop distance from seeds:
  step 0  (  3 nodes)  Activity, Data, Schedule
  step 1  ( 23 nodes)  ActivityMetadataFieldValue, ActivityType, Calendar, IlapFile, ImportStatus, MetadataField, MetadataFieldValue, Profile...
  step 2  ( 33 nodes)  ActivityStructureElement, CalendarOperation, IlapFileProgress, PlanningObjectTypes, PointsInTimeType, ProfilePoint, ReportActivityMetadataFieldValue, ReportActivityMetadataFieldValueArchive...
  step 3  ( 12 nodes)  ConnectedPeriod, Operat

In [7]:
describe(sdo_subgraph, label="SDO")


════════════════════════════════════════════════════════════
  SDO subgraph  —  104 nodes
════════════════════════════════════════════════════════════

By node type:
  attribute          ( 24)  activityCancellationDate, actualFinishDateTime, actualStartDateTime, earlyFinishDateTime, earlyStartDateTime, effortPersonHours...
  concept            ( 46)  ActivityBreakdownStructure, ActualEffortDatum, AlwaysOnScheduleActivity, BaselineSchedule, CancelledActivity, ContinuousWorkPatternAssignment...
  relation           ( 31)  actualFinish, actualStart, earlyFinish, earlyStart, executedOnSite, finishNoEarlierThan...
  seed               (  3)  Project, Schedule, ScheduleActivity

By hop distance from seeds:
  step 0  (  3 nodes)  Project, Schedule, ScheduleActivity
  step 1  ( 51 nodes)  AlwaysOnScheduleActivity, BaselineSchedule, CancelledActivity, EarlyFinishActivity, EarlyStartActivity, LateFinishActivity, LateStartActivity, LiveSchedule...
  step 2  ( 10 nodes)  Duration, EffortDatum, Mi

## Understand rapidfuzz before running the full match

Try a few pairs manually to understand what the scores mean.

In [8]:
from rapidfuzz import fuzz

pairs = [
    # identical concept, same name
    ("earlyStart",           "earlyStart"),
    # identical concept, one has DateTime suffix
    ("earlyStart",           "earlyStartDateTime"),
    # same concept, different naming convention
    ("cancelledDate",        "activityCancellationDate"),
    # same concept, completely different names
    ("plannedWorkHours",     "hasPlannedEffort"),
    # word order difference
    ("startNoEarlierThan",   "noEarlierThanStart"),
    # unrelated
    ("earlyStart",           "lagHours"),
]

print(f"{'GraphQL':<30} {'SDO':<30} {'ratio':>6} {'partial':>8} {'token_sort':>11}")
print("-" * 80)
for a, b in pairs:
    print(
        f"{a:<30} {b:<30}"
        f"{fuzz.ratio(a.lower(), b.lower()):>6.0f}"
        f"{fuzz.partial_ratio(a.lower(), b.lower()):>8.0f}"
        f"{fuzz.token_sort_ratio(a.lower(), b.lower()):>11.0f}"
    )

GraphQL                        SDO                             ratio  partial  token_sort
--------------------------------------------------------------------------------
earlyStart                     earlyStart                       100     100        100
earlyStart                     earlyStartDateTime                71     100         71
cancelledDate                  activityCancellationDate          59      69         59
plannedWorkHours               hasPlannedEffort                  56      69         56
startNoEarlierThan             noEarlierThanStart                72      84         72
earlyStart                     lagHours                          33      43         33


## run the full match

In [9]:
from matching.matcher import match, summary

results = match(gql_subgraph, sdo_subgraph)
summary(results)


════════════════════════════════════════════════════════════
  MATCH RESULTS
════════════════════════════════════════════════════════════
  identical   : 25
  equivalent  : 41
  unique_gql  : 9
  unique_sdo  : 81

── Top identical matches ──
  Activity                            ↔  ScheduleActivity                     score=100.0
  Schedule                            ↔  Schedule                             score=100.0
  Successor                           ↔  hasSuccessor                         score=100.0
  ResourceAssignment                  ↔  ScheduleResourceAssignment           score=100.0
  StructureElement                    ↔  hasBreakdownStructureElement         score=100.0
  ReportSchedule                      ↔  Schedule                             score=100.0
  ScheduleMetadataFieldValue          ↔  Schedule                             score=100.0
  ReportScheduleArchive               ↔  Schedule                             score=100.0
  Profile                            

## explore results as a DataFrame

In [10]:
from matching.matcher import to_dataframe

df = to_dataframe(results)
print(f"Total rows: {len(df)}")
df.head(10)

Total rows: 156


,bucket,gql_name,sdo_name,score,token_sort,partial,ratio
0,identical,Activity,ScheduleActivity,100.0,66.666667,100.0,66.666667
1,identical,Schedule,Schedule,100.0,100.000000,100.0,100.000000
2,identical,Successor,hasSuccessor,100.0,85.714286,100.0,85.714286
3,identical,ResourceAssignment,ScheduleResourceAssignment,100.0,81.818182,100.0,81.818182
4,identical,StructureElement,hasBreakdownStructureElement,100.0,72.727273,100.0,72.727273
5,identical,ReportSchedule,Schedule,100.0,72.727273,100.0,72.727273
6,identical,ScheduleMetadataFieldValue,Schedule,100.0,47.058824,100.0,47.058824
7,identical,ReportScheduleArchive,Schedule,100.0,55.172414,100.0,55.172414
8,identical,Profile,hasResourceProfile,100.0,56.000000,100.0,56.000000
9,identical,Resource,ScheduleResource,100.0,66.666667,100.0,66.666667


In [11]:
# how many in each bucket?
df["bucket"].value_counts()

bucket
unique_sdo    81
equivalent    41
identical     25
unique_gql     9
Name: count, dtype: int64

In [12]:
# identical matches only — sorted by score
df[df["bucket"] == "identical"].sort_values("score", ascending=False)

,bucket,gql_name,sdo_name,score,token_sort,partial,ratio
0,identical,Activity,ScheduleActivity,100.0,66.666667,100.0,66.666667
1,identical,Schedule,Schedule,100.0,100.000000,100.0,100.000000
2,identical,Successor,hasSuccessor,100.0,85.714286,100.0,85.714286
3,identical,ResourceAssignment,ScheduleResourceAssignment,100.0,81.818182,100.0,81.818182
4,identical,StructureElement,hasBreakdownStructureElement,100.0,72.727273,100.0,72.727273
5,identical,ReportSchedule,Schedule,100.0,72.727273,100.0,72.727273
6,identical,ScheduleMetadataFieldValue,Schedule,100.0,47.058824,100.0,47.058824
7,identical,ReportScheduleArchive,Schedule,100.0,55.172414,100.0,55.172414
8,identical,Profile,hasResourceProfile,100.0,56.000000,100.0,56.000000
9,identical,Resource,ScheduleResource,100.0,66.666667,100.0,66.666667


In [13]:
# equivalent matches — these need SME review
df[df["bucket"] == "equivalent"].sort_values("score", ascending=False)

,bucket,gql_name,sdo_name,score,token_sort,partial,ratio
25,equivalent,ReportResourceAssignment,ScheduleResourceAssignment,88.4,76.000000,88.372093,76.000000
26,equivalent,ReportActivity,nextActivity,85.7,76.923077,85.714286,76.923077
27,equivalent,SuccessorType,hasSuccessor,85.7,72.000000,85.714286,72.000000
28,equivalent,ReportSuccessor,hasSuccessor,85.7,66.666667,85.714286,66.666667
29,equivalent,CalendarOperation,Duration,85.7,56.000000,85.714286,56.000000
30,equivalent,ReportStructureElement,hasBreakdownStructureElement,84.2,76.000000,84.210526,76.000000
31,equivalent,IlapFileProgress,hasProgress,84.2,66.666667,84.210526,66.666667
32,equivalent,WeeklyRepeatingPeriod,WeeklyRepeatingWorkPatternAssignment,84.2,66.666667,84.210526,66.666667
33,equivalent,ResourceAssignmentMetadataFieldValue,ScheduleResourceAssignment,81.8,58.064516,81.818182,58.064516
34,equivalent,StructureType,ScheduleBreakdownStructure,81.8,46.153846,81.818182,46.153846


In [14]:
# unique to GraphQL — no SDO counterpart found
df[df["bucket"] == "unique_gql"]

,bucket,gql_name,sdo_name,score,token_sort,partial,ratio
66,unique_gql,ConnectedPeriod,None,57.1,NaN,NaN,NaN
67,unique_gql,MetadataField,None,56.0,NaN,NaN,NaN
68,unique_gql,PointsInTimeType,None,56.0,NaN,NaN,NaN
69,unique_gql,ReportBaselineRevision,None,57.1,NaN,NaN,NaN
70,unique_gql,ReportCalendar,None,59.3,NaN,NaN,NaN
71,unique_gql,ServiceLevelEnum,None,56.0,NaN,NaN,NaN
72,unique_gql,TransferMetadata,None,57.1,NaN,NaN,NaN
73,unique_gql,TypeCode,None,57.1,NaN,NaN,NaN
74,unique_gql,UUID,None,50.0,NaN,NaN,NaN


In [15]:
# unique to SDO — no GraphQL counterpart found
df[df["bucket"] == "unique_sdo"]

,bucket,gql_name,sdo_name,score,token_sort,partial,ratio
75,unique_sdo,None,ActivityBreakdownStructure,NaN,NaN,NaN,NaN
76,unique_sdo,None,ActualEffortDatum,NaN,NaN,NaN,NaN
77,unique_sdo,None,AlwaysOnScheduleActivity,NaN,NaN,NaN,NaN
78,unique_sdo,None,BaselineSchedule,NaN,NaN,NaN,NaN
79,unique_sdo,None,ContinuousWorkPatternAssignment,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
151,unique_sdo,None,startNoLaterThanDateTime,NaN,NaN,NaN,NaN
152,unique_sdo,None,totalFloat,NaN,NaN,NaN,NaN
153,unique_sdo,None,totalFloatHours,NaN,NaN,NaN,NaN
154,unique_sdo,None,totalFloatMinutes,NaN,NaN,NaN,NaN


## adjust thresholds and rerun

The default thresholds are `identical >= 92` and `equivalent >= 60`.
Try different values and see how the bucket sizes change.

In [16]:
results_strict = match(
    gql_subgraph,
    sdo_subgraph,
    threshold_identical=95,
    threshold_equivalent=70,
)
summary(results_strict)


════════════════════════════════════════════════════════════
  MATCH RESULTS
════════════════════════════════════════════════════════════
  identical   : 25
  equivalent  : 29
  unique_gql  : 21
  unique_sdo  : 85

── Top identical matches ──
  Activity                            ↔  ScheduleActivity                     score=100.0
  Schedule                            ↔  Schedule                             score=100.0
  Successor                           ↔  hasSuccessor                         score=100.0
  ResourceAssignment                  ↔  ScheduleResourceAssignment           score=100.0
  StructureElement                    ↔  hasBreakdownStructureElement         score=100.0
  ReportSchedule                      ↔  Schedule                             score=100.0
  ScheduleMetadataFieldValue          ↔  Schedule                             score=100.0
  ReportScheduleArchive               ↔  Schedule                             score=100.0
  Profile                           

## export to Excel for SME review

In [17]:
output_path = ROOT / "mapping_candidates.xlsx"

df.to_excel(output_path, index=False)
print(f"Saved to {output_path}")

Saved to C:\Users\508719\OneDrive - Aker Solutions\Desktop\ILAP\ilap_sdo_mapping_test\mapping_candidates.xlsx


## what comes next

Rapidfuzz catches lexical matches — same or similar names.
It misses conceptual matches where the names are completely different:

```
plannedWorkHours  ↔  hasPlannedEffort     → low lexical score, but same concept
cancelledDate     ↔  activityCancellationDate  → medium score, same concept
```

The next step is `sentence-transformers` — semantic embedding matching.
It embeds `field name + description` as a vector and compares meaning,
not spelling. That will re-examine the `equivalent` and `unique_gql` buckets
and surface matches that rapidfuzz missed.

## run semantic match on the equivalent bucket
### (the ones rapidfuzz wasn't sure about)

In [18]:

from matching.semantic_matcher import semantic_match, summary as semantic_summary

# re-examine only the equivalent bucket from rapidfuzz
equivalent_names = [r["gql"]["name"] for r in results["equivalent"]]

semantic_results = semantic_match(
    gql_subgraph,
    sdo_subgraph,
    candidates=equivalent_names,   # ← focus on uncertain cases
    threshold=0.60,  #adjust here based on how many matches you want to surface
)
semantic_summary(semantic_results)


Loading model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 41 GraphQL nodes ...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Encoding 104 SDO nodes ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
  SEMANTIC MATCH RESULTS
════════════════════════════════════════════════════════════
  matched   : 2
  unmatched : 39

── Top semantic matches ──
  WeeklyRepeatingPeriod               ↔  WeeklyRepeatingWorkPatternAssignment  score=0.7449
  ReportLiveRevision                  ↔  LiveSchedule                         score=0.6263

── Unmatched (no semantic equivalent found) ──
  ActivityMetadataFieldValue           best=hasWorkPattern                  score=0.3477
  ActivityStructureElement             best=nextActivity                    score=0.5521
  ActivityType                         best=ScheduleActivity                score=0.5638
  Calendar                             best=Schedule                        score=0.4702
  CalendarOperation                    best=Schedule                        score=0.5006
  Data                                 best=ScheduleBreakdownStructure      score=0.2729
  IlapFile                

### also run on unique_gql to find hidden matches

In [19]:

unique_names = [r["name"] for r in results["unique_gql"]]

semantic_results_unique = semantic_match(
    gql_subgraph,
    sdo_subgraph,
    candidates=unique_names,
    threshold=0.70,   # slightly looser for unmatched items
)
semantic_summary(semantic_results_unique)

Loading model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding 9 GraphQL nodes ...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding 104 SDO nodes ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


════════════════════════════════════════════════════════════
  SEMANTIC MATCH RESULTS
════════════════════════════════════════════════════════════
  matched   : 1
  unmatched : 8

── Top semantic matches ──
  ReportBaselineRevision              ↔  BaselineSchedule                     score=0.7067

── Unmatched (no semantic equivalent found) ──
  ConnectedPeriod                      best=Duration                        score=0.3161
  MetadataField                        best=hasWorkPattern                  score=0.2226
  PointsInTimeType                     best=finishNoEarlierThan             score=0.4091
  ReportCalendar                       best=Schedule                        score=0.4944
  ServiceLevelEnum                     best=hasResourceProfilePoint         score=0.3087
  TransferMetadata                     best=ResourceBreakdownStructure      score=0.1848
  TypeCode                             best=workingHoursADay                score=0.2701
  UUID                        